# 03 — Conclusões da análise exploratória (D33)

Respostas às perguntas do escopo (`docs/escopo_mvp.md`), cada uma com o número que a sustenta e a
ressalva de `docs/limitacoes.md`. Os números abaixo são recalculados das views da GOLD a cada execução.

Os dados vêm só das views do schema `gold`, lidas como `bed_readonly` pelas funções de `brasil_em_dados.analise`. Os valores aparecem como estão na GOLD (R\$, habitantes, %): nenhuma conversão de unidade (regra 11); os eixos só abreviam o texto dos rótulos.

In [1]:
%matplotlib inline

from brasil_em_dados.analise import calculos, consultas
from brasil_em_dados.analise import graficos as g
from brasil_em_dados.db import engine_readonly

engine = engine_readonly()  # bed_readonly; cada consulta roda numa transação READ ONLY
g.configurar_tabelas()  # números das tabelas no padrão brasileiro

In [2]:
part_pop = consultas.participacao(engine, "pop_censo", 2022)
part_pib = consultas.participacao(engine, "pib_corrente", 2023)
part_pib_2010 = consultas.participacao(engine, "pib_corrente", 2010)
censo_sc = calculos.variacao_entre_anos(consultas.agregado(engine, "estado", "pop_censo"), 2010, 2022)
var = consultas.variacao_censo(engine)
resumo_var = calculos.resumo(var["variacao_percentual"])
pib_sc = consultas.agregado(engine, "estado", "pib_corrente").set_index("ano")["valor"]
per_capita = consultas.painel(engine, 2023)
resumo_pc = calculos.resumo(per_capita["pib_per_capita_reais"])

## 1. Quais municípios concentram população e PIB, e quanto representam do total estadual?

In [3]:
print(f"População (Censo 2022): 10 maiores = {g.formatar_percentual(calculos.participacao_acumulada(part_pop))}; "
      f"metade em {calculos.quantos_somam(part_pop)} municípios. Maior: "
      f"{part_pop['nome_municipio'][0]} ({g.formatar_percentual(part_pop['participacao_percentual'][0])}).")
print(f"PIB nominal (2023): 10 maiores = {g.formatar_percentual(calculos.participacao_acumulada(part_pib))}; "
      f"metade em {calculos.quantos_somam(part_pib)} municípios. Maior: "
      f"{part_pib['nome_municipio'][0]} ({g.formatar_percentual(part_pib['participacao_percentual'][0])}).")

População (Censo 2022): 10 maiores = 40,6%; metade em 17 municípios. Maior: Joinville (8,1%).
PIB nominal (2023): 10 maiores = 46,1%; metade em 13 municípios. Maior: Joinville (9,7%).


**Resposta.** Joinville lidera em população (8,1%) e em PIB (9,7%). Os dez maiores municípios reúnem
cerca de 41% da população e 46% do PIB; metade da população está em 17 municípios e metade do PIB em
13. Itajaí é o 5º em população e o 2º em PIB.

**Ressalva.** O PIB municipal é uma distribuição do PIB estadual feita pelo IBGE por indicadores de
atividade, e não uma medição direta (limitações, seção 6).

## 2. Como a população de cada município variou entre os Censos de 2010 e 2022?

In [4]:
print(f"SC: {g.formatar_percentual(censo_sc['variacao_percentual'][0])} "
      f"({g.formatar_habitantes(censo_sc['valor_inicial'][0])} → {g.formatar_habitantes(censo_sc['valor_final'][0])}).")
print(f"Municípios: {resumo_var.n} comparáveis; {resumo_var.n - resumo_var.negativos} cresceram e "
      f"{resumo_var.negativos} perderam população; mediana {g.formatar_percentual(resumo_var.mediana)}.")
maior_ganho = calculos.top_n(var, "variacao_absoluta", 1).iloc[0]
maior_pct = calculos.top_n(var, "variacao_percentual", 1).iloc[0]
print(f"Maior ganho absoluto: {maior_ganho['nome_municipio']} (+{g.formatar_habitantes(maior_ganho['variacao_absoluta'])}); "
      f"maior ganho relativo: {maior_pct['nome_municipio']} ({g.formatar_percentual(maior_pct['variacao_percentual'])}).")

SC: 21,8% (6,25 mi → 7,61 mi).
Municípios: 293 comparáveis; 223 cresceram e 70 perderam população; mediana 6,8%.
Maior ganho absoluto: Florianópolis (+116,0 mil); maior ganho relativo: Itapoá (108,3%).


**Resposta.** SC cresceu 21,8% entre os Censos (de 6,25 para 7,61 milhões). Dos 293 municípios com
os dois Censos, 223 cresceram e 70 perderam população; a variação mediana foi de 6,8%. As regiões
intermediárias de Florianópolis (+33,8%) e Blumenau (+30,9%) cresceram mais; das 70 perdas, 36 estão
na região de Chapecó e 14 na de Lages.

**Ressalvas.** Pescaria Brava e Balneário Rincão não têm Censo 2010 (instalados em 2013). A queda de
Laguna (−17%) pode vir do desmembramento de Pescaria Brava, mas essa origem é **NÃO CONFIRMADA**
(limitações, seção 10). O Censo 2022 tem duas
apurações, e a diferença entre elas não foi confirmada (seção 8).

## 3. Como o PIB nominal de cada município evoluiu entre 2010 e 2023?

In [5]:
razao = f"{pib_sc[2023] / pib_sc[2010]:.2f}".replace(".", ",")
print(f"PIB nominal de SC: {g.formatar_reais(pib_sc[2010])} → {g.formatar_reais(pib_sc[2023])} "
      f"({razao} vezes, valores correntes).")
print(f"10 maiores: {g.formatar_percentual(calculos.participacao_acumulada(part_pib_2010))} do PIB em 2010 e "
      f"{g.formatar_percentual(calculos.participacao_acumulada(part_pib))} em 2023.")

PIB nominal de SC: R$ 153,7 bi → R$ 513,4 bi (3,34 vezes, valores correntes).
10 maiores: 48,6% do PIB em 2010 e 46,1% em 2023.


**Resposta.** Em valores correntes, o PIB de SC ficou 3,3 vezes maior (de R\$ 153,7 bi para
R\$ 513,4 bi). A participação no total do estado não depende da inflação, por isso mede a mudança
relativa entre municípios: Itajaí subiu de 6,5% para 9,4%, Joinville caiu de 11,9% para 9,7%, e os dez maiores
passaram de 48,6% para 46,1% do PIB.

**Ressalvas.** PIB nominal não é crescimento real (regra 3). 2021–2023 foram revisados em 19/12/2025 e
2022–2023 ainda podem mudar (limitações, seção 5). Em municípios pequenos, variações de um ano podem
passar de 50% pela entrada ou saída de um único estabelecimento (seção 6, alerta A1).

## 4. Como os municípios se comparam em PIB per capita em um mesmo ano?

In [6]:
maior = calculos.top_n(per_capita, "pib_per_capita_reais", 1).iloc[0]
menor = calculos.top_n(per_capita, "pib_per_capita_reais", 1, maiores=False).iloc[0]
razao = f"{resumo_pc.maximo / resumo_pc.minimo:.1f}".replace(".", ",")
print(f"2023: mediana {g.formatar_reais(resumo_pc.mediana)}; maior {maior['nome_municipio']} "
      f"({g.formatar_reais(maior['pib_per_capita_reais'])}); menor {menor['nome_municipio']} "
      f"({g.formatar_reais(menor['pib_per_capita_reais'])}); razão de {razao} vezes.")

2023: mediana R$ 52.591; maior Araquari (R$ 218.626); menor Pescaria Brava (R$ 16.142); razão de 13,5 vezes.


**Resposta.** Em 2023, o PIB per capita mediano foi de R\$ 52,6 mil. O maior (Araquari, R\$ 218,6 mil)
é 13,5 vezes o menor (Pescaria Brava, R\$ 16,1 mil). Seis dos dez maiores têm menos de 10 mil
habitantes (Censo 2022).

**Ressalvas.** PIB per capita não é renda, e o denominador muda conforme o ano (limitações, seção 9).
Em municípios pequenos, a entrada ou a saída de um estabelecimento pode multiplicar o PIB de um ano
para outro; a causa de cada caso é NÃO CONFIRMADA (seção 6).

## 5. Composição setorial

Fora do MVP (decisão D10): a abertura setorial termina em 2021 e fica como extensão
(limitações, seção 7).

## Pergunta central

*Como os municípios de Santa Catarina se diferenciam em população e dimensão econômica, e como esses
indicadores evoluíram entre 2010 e 2023?*

Nenhum município passa de 10% da população (máximo: Joinville, 8,1% em 2022) nem do PIB (máximo:
Joinville, 9,7% em 2023). Os dez maiores ficam abaixo de metade dos dois totais (40,6% da população
e 46,1% do PIB) e estão em 6 das 7 regiões intermediárias. Entre os Censos, as regiões
intermediárias de Florianópolis (+33,8%) e Blumenau (+30,9%) cresceram mais, e 50 dos 70 municípios
que perderam população estão nas regiões de Chapecó e de Lages. No PIB, a participação dos dez
maiores de 2023 caiu de 48,6% para 46,1%: Itajaí ganhou 2,8 pontos, e Joinville e Florianópolis
perderam. O PIB per capita de 2023 varia 13,5 vezes entre municípios; seis dos dez maiores têm menos
de 10 mil habitantes, e, em municípios pequenos, a entrada ou a saída de um estabelecimento pode
multiplicar o PIB (limitações, seção 6).

Tudo isso vale com as ressalvas acima: PIB nominal, PIB per capita que não é renda, Censo e
estimativa separados, e os dois municípios instalados em 2013.